# 02 — Autograd and Gradients

This notebook builds intuition for PyTorch's automatic differentiation engine.

By the end, you will be able to:
- explain computational graphs
- compute gradients with `.backward()`
- reset gradients correctly
- perform a manual gradient descent step

In [ ]:
import torch

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)

## 1) Leaf tensors and `requires_grad`

A tensor tracks gradients when `requires_grad=True`.

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = 3 * x**2 + 2 * x + 1  # y = 3x^2 + 2x + 1

print("x:", x)
print("y:", y)
print("is y part of graph?", y.grad_fn is not None)

## 2) Backpropagation with `.backward()`

For scalar output `y`, calling `y.backward()` computes $\frac{dy}{dx}$.

In [ ]:
y.backward()
print("x.grad:", x.grad.item())
print("Expected derivative at x=2 -> 6x + 2 =", 6*2 + 2)

## 3) Gradient accumulation

Gradients accumulate in `.grad`. Always clear them before next backward pass in training loops.

In [ ]:
x = torch.tensor([1.0], requires_grad=True)

for step in range(3):
    y = x**2
    y.backward()
    print(f"step={step}, grad={x.grad.item()}")

print("\nNow with zeroing gradients each step:")
x = torch.tensor([1.0], requires_grad=True)
for step in range(3):
    if x.grad is not None:
        x.grad.zero_()
    y = x**2
    y.backward()
    print(f"step={step}, grad={x.grad.item()}")

## 4) Non-scalar outputs need `grad_outputs`

If the output has more than one value, provide a vector of upstream gradients.

In [ ]:
v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = v**2  # vector output

upstream = torch.tensor([1.0, 1.0, 1.0])
out.backward(gradient=upstream)
print("v.grad:", v.grad)
print("Expected:", 2*v.detach())

## 5) Manual gradient descent on a tiny linear model

We fit $y = wx + b$ to synthetic data with manual updates.

In [ ]:
x = torch.linspace(-2, 2, 100).unsqueeze(1)
true_w = torch.tensor([[2.5]])
true_b = torch.tensor([0.7])
y = x @ true_w + true_b + 0.2 * torch.randn_like(x)

w = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

lr = 0.05
for epoch in range(200):
    pred = x @ w + b
    loss = ((pred - y) ** 2).mean()

    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    w.grad.zero_()
    b.grad.zero_()

    if (epoch + 1) % 50 == 0:
        print(f"epoch={epoch+1:>3}, loss={loss.item():.4f}")

print("\nLearned parameters:")
print("w:", w.item(), "(true:", true_w.item(), ")")
print("b:", b.item(), "(true:", true_b.item(), ")")

## 6) Exercises

1. Change the learning rate to `0.5`, then to `0.005`. What happens to convergence?
2. Replace MSE with MAE (`torch.abs(pred-y).mean()`). Compare learned parameters.
3. Add a second feature to `x` and learn a 2D linear regression model.

When you are done, continue to `03_training_loop_from_scratch.ipynb`.